# Concatenating, Joining, and Pivoting Data

---
title: "PA 3.1: Concatenating, Joining, and Pivoting"
format:
  html:
    embed-resources: true
---

There are several ways of answering many questions in this notebook, but try to use the new operations (concatenate, joint/merge, pivot) whenever possible to get practice.

In [3]:
import pandas as pd

## Movie Ratings Data

The Movielens data set (https://dlsun.github.io/pods/data/ml-1m/ ) contains 1 million movie ratings submitted by users. The information about the movies, ratings, and users are stored in three separate files, called `movies.dat`, `ratings.dat`, and `users.dat`. The column names are not included with the data files. Refer to the webpage above for more information.


1\. Read in each of the data files.


(Hint: see the note in the documentation about the delimiter; use the `sep` argument in `read_csv`. If you get an error when reading the `movies.dat` file, try `encoding_errors = "ignore"` in `read_csv`.)

In [4]:
base = "https://dlsun.github.io/pods/data/ml-1m/"

movies_url = base + "movies.dat"
movies_df = pd.read_csv(movies_url,sep = "::", encoding_errors="ignore", header =None, names = ["movies_id","title","genre"])

<positron-console-cell-4>:4: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.


In [5]:
ratings_url = base + "ratings.dat"

ratings_df = pd.read_csv(
    ratings_url,
    sep="::",
    engine="python",
    header=None,
    names=["user_id", "movies_id", "rating", "timestamp"])

In [6]:
users_url = base + "users.dat"

users_df = pd.read_csv(
    users_url,
    sep="::",
    engine="python",
    header=None,
    names=["user_id", "gender", "age", "occupation", "zip_code"]
)

In [7]:
ratings_df.head()
users_df.head()

,user_id,gender,age,occupation,zip_code
0,1,F,1,10,48067
1,2,M,56,16,70072
2,3,M,25,15,55117
3,4,M,45,7,02460
4,5,M,25,20,55455


In [8]:
movies_df.head()

,movies_id,title,genre
0,1,Toy Story (1995),Animation|Children's|Comedy
1,2,Jumanji (1995),Adventure|Children's|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama
4,5,Father of the Bride Part II (1995),Comedy


2\. Which age group tends to give the highest ratings? Create an appropriate summary to answer this question.

(Note the way age is coded in the [REAMDE file](https://dlsun.github.io/pods/data/ml-1m/README.txt).)

In [9]:
ratings_users = ratings_df.merge(
    users_df,
    on="user_id",
    how="inner")

In [10]:
ratings_users_df = ratings_df.merge(
    users_df,
    on="user_id",
    how="left", validate="many_to_one")

age_label = {1:"<18", 18:"18-24", 25:"25-34", 35:"35-44", 45:"45-49", 50:"50-55", 56:"56+"}
ratings_users_df["age_group"] = ratings_users_df["age"].map(age_label)

(ratings_users_df.groupby("age_group")["rating"]
.agg(n_rating = "size", mean_rating = "mean")
.sort_values("mean_rating", ascending=False)
.reset_index()
)

,age_group,n_rating,mean_rating
0,56+,38780,3.766632
1,50-55,72490,3.714512
2,45-49,83633,3.638062
3,35-44,199003,3.618162
4,<18,27211,3.549520
5,25-34,395556,3.545235
6,18-24,183536,3.507573


3\. Among movies with at least 100 ratings, which movies had the highest average rating? The lowest?

In [11]:
movie_summary = (
    ratings_df
    .groupby("movies_id")["rating"]
    .agg(["size", "mean"]))

movie_100 = movie_summary[movie_summary["size"] >= 100]

In [12]:
# way it was done in class, needs revision
sum_ratings_df = ratings_df.groupby("movies_id")["rating"].agg(n_ratings = "size", mean_ratings = "mean").resetindex()
movies_ratings_df = sum_ratings_df.merge(movies_df, on ="movies_id", how = "left")

AttributeError: 'DataFrame' object has no attribute 'resetindex'

In [13]:
# best rated movies
movie_100.sort_values("mean", ascending=False).head()

,size,mean
movies_id,,
2019,628,4.560510
318,2227,4.554558
858,2223,4.524966
745,657,4.520548
50,1783,4.517106


In [14]:
# Worst rated movies
movie_100.sort_values("mean").head()

,size,mean
movies_id,,
810,120,1.466667
3593,342,1.611111
3799,100,1.620000
2817,125,1.640000
2383,149,1.657718


In [15]:
movie_100 = movie_100.merge(
    movies_df,
    on="movies_id")

4\. For each movie, calculate the average rating and the proportion of the ratings that were from users aged 18-24. Make a scatterplot showing the relationship between the average rating and the 18-24 share, with each point representing a movie. (Optional: Use the size of each point to represent the number of users who rated the movie.)

In [16]:
movie_summary = (
    ratings_users_df
    .groupby("movies_id")
    .agg(
        avg_rating=("rating", "mean"),
        prop_18_24=("age_group", lambda s: (s == "18-24").mean()))
    .reset_index())

In [17]:
movie_summary.head()

,movies_id,avg_rating,prop_18_24
0,1,4.146846,0.215696
1,2,3.201141,0.219686
2,3,3.016736,0.223849
3,4,2.729412,0.229412
4,5,3.006757,0.219595


5\. Calculate the number of ratings by movie. How many of the movies had zero ratings?

(_Hint_: Why is an inner join not sufficient here?)

In [18]:
movies_ratings = movies_df.merge(
    ratings_df,
    on="movies_id",
    how="left")

In [19]:
rating_counts = (
    movies_ratings
    .groupby(["movies_id", "title"])["rating"]
    .count()
    .reset_index())

In [20]:
len(rating_counts[rating_counts["rating"] == 0])

177

6\. How many movies received both a 1 and a 5 rating? Which movies? How many ratings of each type (1 or 5 stars)? Answer these question by joining two appropriate tables.

In [21]:
ratings_1 = (
    ratings_df[ratings_df["rating"] == 1]
    .groupby("movies_id")["rating"]
    .count()
    .reset_index())

In [22]:
ratings_1 = ratings_1.rename(
    columns={"rating": "count_1"})

In [23]:
ratings_5 = (
    ratings_df[ratings_df["rating"] == 5]
    .groupby("movies_id")["rating"]
    .count()
    .reset_index())

In [24]:
ratings_5 = ratings_5.rename(
    columns={"rating": "count_5"})

In [25]:
both_ratings = ratings_1.merge(
    ratings_5,
    on="movies_id",
    how="inner")

In [26]:
both_ratings = both_ratings.merge(
    movies_df,
    on="movies_id",
    how="left")

In [27]:
both_ratings[
    ["movies_id", "title", "count_1", "count_5"]]

,movies_id,title,count_1,count_5
0,1,Toy Story (1995),16,820
1,2,Jumanji (1995),42,48
2,3,Grumpier Old Men (1995),44,43
3,4,Waiting to Exhale (1995),21,6
4,5,Father of the Bride Part II (1995),28,16
...,...,...,...,...
2981,3948,Meet the Parents (2000),35,163
2982,3949,Requiem for a Dream (2000),9,132
2983,3950,Tigerland (2000),2,12
2984,3951,Two Family House (2000),1,14


In [ ]:
# 2,986 movies received both at least one 1 star rating and at least one 5 star rating